# TOPOTEX Orientation-Aligned Dataset Inspector

检查当前 OA prototype 数据（`topotex_data_OA`）：canonical mesh +
stochastic rendered observations + native / xatlas / connected-partial
三查询。**唯一数据根为 OA_ROOT**；相机参数只是 debug 元数据，绝不作为
模型输入。

In [ ]:
from pathlib import Path

OA_ROOT = Path("/root/youjiaZhang/topotex_data_OA")
RANDOM_SAMPLE = True
SAMPLE_ID = None            # 指定对象 id 时置 RANDOM_SAMPLE=False
RANDOM_SEED = 20260801
N_SHOW = 16
MODE = "quick"              # "quick" | "full"（full: 一致性检查扫全部对象）

In [ ]:
import hashlib, json, os
import numpy as np
import matplotlib.pyplot as plt
import trimesh
from PIL import Image
from safetensors.numpy import load_file

DS = OA_ROOT / "dataset"
man = [json.loads(l) for l in open(DS / "manifest.jsonl")]
uids = [m["sample_id"] for m in man]
cat_of = {m["sample_id"]: m.get("lvis_category") for m in man}
rng = np.random.default_rng(RANDOM_SEED if RANDOM_SAMPLE else None)
print(f"OA_ROOT = {OA_ROOT} | objects = {len(uids)} | MODE = {MODE}")

## Section 0 — OA Dataset Overview

In [ ]:
sel = json.load(open(OA_ROOT / "oa100_selected.json"))
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
try:
    import subprocess
    commit = subprocess.check_output(
        ["git", "-C", "/root/youjiaZhang/TopoTex", "rev-parse", "--short", "HEAD"],
        text=True).strip()
except Exception:
    commit = "n/a"
metas = {u: json.loads((DS / "samples" / u / "meta.json").read_text()) for u in uids}
faces = np.array([metas[u]["num_faces"] for u in uids])
verts = np.array([metas[u]["num_vertices"] for u in uids])
src_tex = [tuple(metas[u]["source_texture_shape"]) for u in uids]
vt = np.array([int(load_file(str(DS / "samples" / u / "uv_address.safetensors"))["valid_mask"].sum()) for u in uids])
print(f"candidate count        {sel['n_candidates']}")
print(f"passed-gate count      {sel['n_gate_pass']}")
print(f"selected objects       {sel['n_selected']}  ({sel['selection_rule']})")
print(f"manifest SHA-256       {sha(DS / 'manifest.jsonl')}")
print(f"inspection code commit {commit}")
print(f"stored texture res     256x256 (source textures: {len(set(src_tex))} distinct shapes, e.g. {sorted(set(src_tex))[-3:]})")
print(f"native UV valid texels min {vt.min()} / median {int(np.median(vt))} / max {vt.max()}  (of 65536)")
cats = [cat_of[u] or "unknown" for u in uids]
top = sorted(((c, cats.count(c)) for c in set(cats)), key=lambda t: -t[1])
fig, axes = plt.subplots(1, 3, figsize=(16, 3.4))
axes[0].barh([c for c, _ in top[:15]][::-1], [n for _, n in top[:15]][::-1])
axes[0].set_title(f"LVIS categories ({len(top)} distinct, top 15)", fontsize=9)
axes[1].hist(faces, bins=24); axes[1].set_title(f"face count (min {faces.min()} / med {int(np.median(faces))} / max {faces.max()})", fontsize=9)
axes[2].hist(verts, bins=24); axes[2].set_title(f"vertex count (med {int(np.median(verts))})", fontsize=9)
plt.tight_layout(); plt.show()

## Section 1 — Random OA Object

In [ ]:
UID = SAMPLE_ID or uids[int(rng.integers(len(uids)))]
d = DS / "samples" / UID
h = OA_ROOT / "oa100" / UID
m = metas[UID]
tr = json.loads((h / "mesh/transform.json").read_text())
sc = trimesh.load(str(h / "mesh/canonical.glb"), force="mesh", process=False)
V = np.asarray(sc.vertices); Vn = (V - V.mean(0)) / max(np.abs(V - V.mean(0)).max(), 1e-9)
idx = np.random.default_rng(0).choice(len(Vn), min(5000, len(Vn)), replace=False)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (t, a, b) in zip(axes, [("XY", 0, 1), ("ZY", 2, 1), ("XZ top", 0, 2)]):
    ax.scatter(Vn[idx, a], Vn[idx, b], s=0.5, c="k")
    ax.arrow(0, 0, .45, 0, color="r", width=.008); ax.arrow(0, 0, 0, .45, color="g", width=.008)
    ax.set_title(f"canonical mesh {t}", fontsize=9); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
plt.show()
print(f"object ID            {UID}")
print(f"LVIS category        {cat_of.get(UID)}")
print(f"orientation_aligned  {tr['orientation_aligned']}")
print(f"vertices / faces     {m['num_vertices']} / {m['num_faces']}")
print(f"transform R          {np.array(tr['R']).round(4).tolist()}")
print(f"transform t / s      {tr['t']} / {tr['s']}")
print(f"texture resolution   stored {m['texture_resolution']} (source {m['source_texture_shape']})")

## Section 2 — Stochastic Canonical Views

六张视图全部是 **canonical mesh 的真实渲染**（确定性 rebake）；
azimuth/elevation 只是 debug 标注——**camera metadata 不是模型输入**。

In [ ]:
vm = json.loads((h / "images/view_meta.json").read_text())
fig, axes = plt.subplots(1, 6, figsize=(21, 3.6))
for k in range(6):
    axes[k].imshow(Image.open(h / f"images/view_{k:03d}.png"))
    axes[k].set_title(f"view_{k:03d}  az {vm[k]['azimuth']:.1f}  el {vm[k]['elevation']:.1f}", fontsize=8)
    axes[k].axis("off")
plt.show()
print("camera metadata (debug only):", json.dumps(vm))

## Section 3 — Texture and UV Queries

**partial is a surface subset query, not an unwrap family**（它复用
native layout，只保留一个连通面子集的 texels）。

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9.2))
for r_i, (q, sem) in enumerate([("uv_000", "native full layout"),
                                ("uv_001", "xatlas full layout"),
                                ("uv_002", "connected partial query")]):
    qa = load_file(str(d / f"uv_queries/{q}/uv_address.safetensors"))
    gt = plt.imread(d / f"uv_queries/{q}/gt_texture.png")
    fid = qa["face_id"]; bar = qa["barycentric"].astype(np.float32); val = qa["valid_mask"].astype(bool)
    uvv = qa["uv_vertices"]
    axes[r_i, 0].scatter(uvv[:, 0], 1 - uvv[:, 1], s=0.12, c="k"); axes[r_i, 0].set_aspect("equal")
    axes[r_i, 0].set_title(f"{sem}\n({len(uvv)} uv verts)", fontsize=8)
    axes[r_i, 1].imshow(gt); axes[r_i, 1].set_title("GT texture", fontsize=8)
    axes[r_i, 2].imshow(np.where(val, fid, -1), cmap="nipy_spectral"); axes[r_i, 2].set_title(f"face_id (max {fid.max()})", fontsize=8)
    bar_rgb = np.clip(bar.transpose(1, 2, 0), 0, 1) * val[..., None]
    axes[r_i, 3].imshow(bar_rgb); axes[r_i, 3].set_title("barycentric RGB", fontsize=8)
    axes[r_i, 4].imshow(val, cmap="gray"); axes[r_i, 4].set_title(f"valid mask ({val.mean() * 100:.1f}%)", fontsize=8)
    for ax in axes[r_i]: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## Section 4 — Consistency Checks（自动 PASS/FAIL）

In [ ]:
def check_object(uid):
    dd = DS / "samples" / uid; hh = OA_ROOT / "oa100" / uid
    out = {}
    try:
        ms = load_file(str(dd / "mesh.safetensors")); F = ms["faces"]
        out["mesh readable"] = ms["vertices"].ndim == 2 and len(F) > 0
        out["no NaN/Inf (mesh)"] = bool(np.isfinite(ms["vertices"]).all()) and bool(np.isfinite(ms["uv_vertices"]).all())
        # six stochastic renders (container filename kept for loader compatibility)
        st = load_file(str(dd / "mv.safetensors"))
        out["six views readable"] = st["images"].shape == (6, 3, 256, 256) and st["images"].dtype == np.uint8
        out["texture readable"] = plt.imread(dd / "gt_texture.png").shape[:2] == (256, 256)
        tr_ = json.loads((hh / "mesh/transform.json").read_text())
        out["orientation metadata"] = tr_.get("orientation_aligned") is True and np.array(tr_["R"]).shape == (3, 3)
        out["render non-empty"] = bool(all(np.asarray(Image.open(hh / f"images/view_{k:03d}.png")).std() > 2 for k in range(6)))
        ok_fid = ok_bary = ok_mask = True
        for q in ("uv_000", "uv_001", "uv_002"):
            qa = load_file(str(dd / f"uv_queries/{q}/uv_address.safetensors"))
            v = qa["valid_mask"].astype(bool)
            fidv = qa["face_id"][v]
            bs = qa["barycentric"].astype(np.float32).transpose(1, 2, 0)[v].sum(-1)
            ok_fid &= bool(v.any()) and bool(fidv.min() >= 0) and bool(fidv.max() < len(F))
            ok_bary &= bool(np.abs(bs - 1).max() < 2e-2) and bool(np.isfinite(qa["barycentric"].astype(np.float32)).all())
            ok_mask &= bool((qa["face_id"][~v] == -1).all())
        out["UV face_id range"] = bool(ok_fid)
        out["barycentric sum"] = bool(ok_bary)
        out["mask consistent"] = bool(ok_mask)
        out["same object identity"] = bool(
            os.stat(dd / "gt_texture.png").st_ino == os.stat(hh / "texture/gt_texture.png").st_ino
            and os.stat(dd / "meta.json").st_ino == os.stat(hh / "metadata.json").st_ino)
    except Exception as e:
        out["exception"] = f"FAIL {type(e).__name__}: {e}"
    return out

GALLERY = ([uids[i] for i in rng.choice(len(uids), min(N_SHOW, len(uids)), replace=False)]
           if RANDOM_SAMPLE else uids[:N_SHOW])
scan = uids if MODE == "full" else sorted(set(GALLERY) | {UID})
n_bad = 0
for u in scan:
    r = check_object(u)
    bad = not all(v is True for v in r.values())
    n_bad += bad
    if bad:
        print("!!", u, r)
print(f"consistency: {len(scan) - n_bad}/{len(scan)} objects pass "
      f"({'ALL' if MODE == 'full' else 'gallery+current'} scan); checks/object = {len(check_object(UID))}")
stats = {"n_objects": len(uids), "n_categories": len(set(c for c in cats if c != 'unknown')),
         "face_count": {"min": int(faces.min()), "median": int(np.median(faces)), "max": int(faces.max())},
         "views_per_object": 6, "queries_per_object": 3, "resolution": 256,
         "consistency": {"scanned": len(scan), "failed": n_bad}}
outd = Path(os.environ.get("TOPOTEX_RUN_ROOT", str(OA_ROOT / "runs"))) / "dataset_inspector_oa"
outd.mkdir(parents=True, exist_ok=True)
json.dump(stats, open(outd / "oa_dataset_statistics.json", "w"), indent=1)
print("stats ->", outd / "oa_dataset_statistics.json")

## Section 5 — Random Gallery（N_SHOW 个对象）

In [ ]:
for uid in GALLERY:
    dd = DS / "samples" / uid; hh = OA_ROOT / "oa100" / uid
    mm = metas[uid]
    tr_ = json.loads((hh / "mesh/transform.json").read_text())
    sc_ = trimesh.load(str(hh / "mesh/canonical.glb"), force="mesh", process=False)
    Vv = np.asarray(sc_.vertices); Vv = (Vv - Vv.mean(0)) / max(np.abs(Vv - Vv.mean(0)).max(), 1e-9)
    ii = np.random.default_rng(0).choice(len(Vv), min(3000, len(Vv)), replace=False)
    qa = load_file(str(dd / "uv_queries/uv_000/uv_address.safetensors"))
    fig, axes = plt.subplots(1, 9, figsize=(27, 3.0))
    axes[0].scatter(Vv[ii, 0], Vv[ii, 1], s=0.4, c="k"); axes[0].set_aspect("equal")
    axes[0].set_title("canonical mesh XY", fontsize=8)
    for k in range(6):
        axes[1 + k].imshow(Image.open(hh / f"images/view_{k:03d}.png"))
        axes[1 + k].set_title(f"v{k}", fontsize=8)
    axes[7].imshow(plt.imread(dd / "gt_texture.png")); axes[7].set_title("native GT texture", fontsize=8)
    axes[8].imshow(qa["valid_mask"].astype(bool), cmap="gray")
    axes[8].set_title(f"native UV mask ({qa['valid_mask'].astype(bool).mean() * 100:.0f}%)", fontsize=8)
    for ax in axes: ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"{uid} | {cat_of.get(uid)} | faces {mm['num_faces']} | aligned {tr_['orientation_aligned']}",
                 fontsize=9, y=1.03)
    plt.tight_layout(); plt.show()